In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import sys
from tqdm import tqdm

import numpy as np
import pandas as pd
from pathlib import Path
from skimage.io import imread

%load_ext autoreload
%autoreload 2
# repo_path = '/path/to/PRISM_Code'
# package_path = repo_path / '' 
# if package_path not in sys.path: sys.path.append(package_path)

BASE_DIR = Path('/path/to/processed')
RUN_ID = '20230523_HCC_PRISM_probe_refined'
src_dir = BASE_DIR / f'{RUN_ID}_processed'
stc_dir = src_dir / 'stitched'
read_dir = src_dir / 'readout'
seg_dir = src_dir / 'segmented'

In [ ]:
#read cell centroid
centroids = pd.read_csv(seg_dir/'centroids_all.csv', header=None).to_numpy()
print('centroids_num:',len(centroids))

#read all labeled rna
rna_df = pd.read_csv(seg_dir/'rna_labeled.csv')
rna_df

In [ ]:
# read mask ROI
Mask = imread(visual_dir/'Mask.tif')
Mask.shape

In [ ]:
# reserve nucleus that is in ROI
cell_df = pd.DataFrame(centroids)
cell_df['AP'] = Mask[centroids[:,0],centroids[:,1]]

In [ ]:
# reserve Cell Index that is in ROI
NTS_rna_labeled = rna_df[rna_df['Cell Index'].isin(list(cell_df[cell_df['AP']!=0].index))]

In [ ]:
# Generate Expression matrix that is in ROI
match_df = NTS_rna_labeled.copy()
match_df['Count'] = np.ones(len(match_df))
match_df_group = match_df[['Cell Index','Gene','Count']].groupby(['Cell Index','Gene']).count()
matrix = match_df_group.unstack().fillna(0)
matrix.columns = matrix.columns.droplevel()
matrix.columns.name = None
matrix.index.name = None
matrix = matrix.drop(columns='Other')
matrix['sum'] = matrix.iloc[:,0:57].sum(axis=1)
# filter matrix by count
matrix = matrix[matrix['sum']>=5]
matrix = matrix.drop(columns='sum')
matrix

In [ ]:
# Draw co=expression matrix
import matplotlib.pyplot as plt
from draw_spots import crop_df
from skimage.io import imread
from skimage.io import imsave
import numpy as np
from glob import glob
import os
from tqdm import tqdm
from tqdm import trange
import seaborn as sns
df_pre_corr = matrix
df_pre_corr = df_pre_corr / df_pre_corr.sum()
df_corr = df_pre_corr.corr()
df_corr.values[[np.arange(df_corr.shape[0])]*2] = None
#df_corr = df_corr.sort_values(by='Gfap',ascending=False)
df_corr.to_csv(visual_dir/'df_coor.csv')
sns.set(font_scale=0.2)
ax = sns.heatmap(df_corr,cmap='coolwarm',xticklabels=1,yticklabels=1,vmin=-0.1,vmax=0.5)
plt.savefig(visual_dir/'cell-co-expression.pdf')
plt.show()

In [ ]:
matrix.to_csv(visual_dir/'NTS_expression_matrix.csv')